# 00 — Arm Setup and Reference Animations

**Purpose**: Prepare the SO-ARM101 for coaching sessions. Verifies hardware, calibrates both arms, confirms camera feeds, records reference animations for each task, and replays them to confirm quality.

Run this notebook **once per lab setup** (new location, new calibration) and whenever you want to record fresh reference animations.

**Prereqs**:
- Pi powered on and reachable via SSH
- Leader and follower arms plugged into USB (`/dev/ttyACM0`, `/dev/ttyACM1`)
- Cameras connected (`/dev/video0`, `/dev/video2`)
- `.env` populated with `PI_HOST`, `PI_PORT`, `HF_USER`, `HF_TOKEN`

**Outcome**: Calibration files on Pi, reference datasets on HuggingFace Hub, follower arm verified with replay.

---

In [ ]:
import os
import shlex
import time
import json
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(dotenv_path=Path('..') / '.env', override=False)

_bench = {
    "notebook": "00_arm_setup",
    "started_at": datetime.utcnow().isoformat(),
    "timings": {}, "checks": {}
}
_t0 = time.monotonic()

PI_HOST   = os.getenv("PI_HOST", "192.168.4.191")
PI_PORT   = os.getenv("PI_PORT", "22222")
PI_USER   = "root"
HF_USER   = os.getenv("HF_USER")
HF_TOKEN  = os.getenv("HF_TOKEN")
PI_IMAGE  = f"{HF_USER}/lerobot-soarm101:latest"

# Task reference dataset slugs
TASK_TOUCH_SLUG  = os.getenv("TASK_TOUCH_SLUG",  "touch-block")
TASK_TOUCH_DESC  = os.getenv("TASK_TOUCH_DESC",  "Touch the red block on the table")
TASK_PICK_SLUG   = os.getenv("TASK_PICK_SLUG",   "pick-place-block")
TASK_PICK_DESC   = os.getenv("TASK_PICK_DESC",   "Pick up the red block and place it in the bowl")

def pi_run(cmd, capture=False, stream=False):
    ssh_prefix = [
        "ssh", "-p", PI_PORT, "-o", "StrictHostKeyChecking=no",
        "-o", "ConnectTimeout=10", f"{PI_USER}@{PI_HOST}",
    ]
    full = ssh_prefix + ["bash", "-c", cmd]
    if stream:
        proc = subprocess.Popen(full, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    elif capture:
        r = subprocess.run(full, capture_output=True, text=True)
        return r.stdout.strip() if r.returncode == 0 else ""
    else:
        return subprocess.run(full, capture_output=True, text=True)

print(f"Pi:    {PI_HOST}:{PI_PORT}")
print(f"Image: {PI_IMAGE}")
print(f"Tasks: {TASK_TOUCH_SLUG}, {TASK_PICK_SLUG}")

## 1. Verify Pi and Devices

In [ ]:
_t = time.monotonic()
checks = {}

# Pi SSH
r = subprocess.run(
    ["ssh", "-p", PI_PORT, "-o", "ConnectTimeout=8", "-o", "StrictHostKeyChecking=no",
     f"{PI_USER}@{PI_HOST}", "echo ok"],
    capture_output=True, text=True
)
checks["pi_ssh"] = r.returncode == 0

# Serial ports
serials = pi_run("ls /dev/ttyACM* 2>/dev/null", capture=True)
serial_list = serials.splitlines() if serials else []
checks["serial_ports"] = len(serial_list) >= 2

# Cameras
cams = pi_run("ls /dev/video0 /dev/video2 2>/dev/null", capture=True)
checks["cameras"] = "/dev/video0" in cams

# Container image present
img = pi_run(f"balena images --filter=reference={PI_IMAGE} --format '{{{{.Repository}}}}' 2>/dev/null | head -1", capture=True)
checks["container_image"] = bool(img)

print("=" * 40)
for name, ok in checks.items():
    icon = "OK  " if ok else "FAIL"
    extra = ""
    if name == "serial_ports" and serial_list:
        extra = f"  {', '.join(serial_list)}"
    elif name == "container_image" and not ok:
        extra = f"  (pull with: balena pull {PI_IMAGE})"
    print(f"  [{icon}] {name}{extra}")

_bench["checks"] = checks
_bench["timings"]["device_check_s"] = round(time.monotonic() - _t, 1)

if not all(checks.values()):
    print("\nFix failing checks before continuing.")

## 2. Identify Leader and Follower Ports

The leader arm (free-moving, no torque) and follower arm (torque enabled) can swap ports on reconnect. This cell confirms which is which before calibrating.

**Move the arm plugged into ttyACM0 when prompted.**

In [ ]:
port_check_script = '''
python3 -c "
import time
from lerobot.motors.feetech.feetech import FeetechMotorsBus
from lerobot.motors import Motor, MotorNormMode

motors = {str(i): Motor(i, 'sts3215', MotorNormMode.RANGE_M100_100) for i in range(1,7)}
bus = FeetechMotorsBus(port='/dev/ttyACM0', motors=motors)
bus.connect()
print('Move the ttyACM0 arm now (3s)...')
for i in [3,2,1]: print(f'{i}...'); __import__('time').sleep(1)
pos = bus.sync_read('Present_Position', normalize=False)
vals = list(pos.values())
spread = max(vals) - min(vals)
print(f'Positions: {vals}')
print(f'Spread: {spread} — if > 50, this arm was moved (likely LEADER)')
bus.disconnect()
"
'''

print("Checking ttyACM0 — move that arm when you see the countdown...")
rc = pi_run(
    f"balena run --rm --privileged --device=/dev/ttyACM0 {PI_IMAGE} bash -c '{port_check_script}'",
    stream=True
)

print()
print("NOTE: The arm with free movement (no resistance) is the LEADER.")
print("      Confirm serial numbers with: balena run --rm --privileged")
print("      --device=/dev/ttyACM0 --device=/dev/ttyACM1 {PI_IMAGE}")
print("      python3 -c 'from lerobot.motors.feetech.feetech import FeetechMotorsBus; ...")
print()
print("Known serials (update fleet.yaml if different):")
print("  ttyACM0 = leader  (serial 5970072696)")
print("  ttyACM1 = follower (serial 5970072616)")

## 3. Camera Preview

Verify both cameras are framing the workspace correctly before recording.

In [ ]:
from IPython.display import IFrame, display
import subprocess, time

PREVIEW_PORT = 7860

# Stop any existing preview, start fresh
pi_run("balena stop $(balena ps -q 2>/dev/null) 2>/dev/null || true")
time.sleep(2)

pi_run(
    f"balena run -d --privileged "
    f"--device=/dev/video0 --device=/dev/video2 "
    f"-p {PREVIEW_PORT}:{PREVIEW_PORT} "
    f"{PI_IMAGE} python scripts/camera_preview.py"
)
time.sleep(3)

# Open SSH tunnel to preview
tunnel = subprocess.Popen(shlex.split(
    f"ssh -p {PI_PORT} -L {PREVIEW_PORT}:localhost:{PREVIEW_PORT} "
    f"-N -o StrictHostKeyChecking=no {PI_USER}@{PI_HOST}"
))
time.sleep(2)
print(f"Camera preview: http://localhost:{PREVIEW_PORT}")
display(IFrame(src=f"http://localhost:{PREVIEW_PORT}", width="100%", height=480))

## 4. Calibrate Arms

Calibration maps each joint's physical range to normalized values. Run **once per arm setup** — files are saved to `/mnt/data/calibration/` on the Pi and persist across reboots.

**If calibration files already exist, skip this cell.**

In [ ]:
# Check if calibration files already exist
cal_check = pi_run(
    "ls /mnt/data/calibration/*.json 2>/dev/null | wc -l",
    capture=True
)
cal_count = int(cal_check or 0)

if cal_count >= 2:
    print(f"Calibration files found ({cal_count}): skipping.")
    files = pi_run("ls /mnt/data/calibration/", capture=True)
    print(files)
else:
    print("No calibration files found — calibration required.")
    print()
    print("Stop the camera preview first, then run in your terminal:")
    print()

    # Copy fleet config to Pi for calibration
    subprocess.run(
        shlex.split(f"scp -P {PI_PORT} config/fleet.yaml {PI_USER}@{PI_HOST}:/tmp/fleet.yaml"),
        check=True
    )

    print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
    print(f'  "balena run -it --privileged \\')
    print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
    print(f'   -v /tmp/fleet.yaml:/app/config/fleet.yaml \\')
    print(f'   -v /mnt/data/calibration:/app/calibration \\')
    print(f'   {PI_IMAGE} \\')
    print(f'   coachable --fleet /app/config/fleet.yaml calibrate --robot alpha"')
    print()
    print("Move each joint through its FULL range when prompted.")
    print("Re-run this cell after calibration completes to verify.")

## 5. Verify Teleoperation

Before recording, confirm the follower arm tracks the leader arm smoothly.

In [ ]:
print("Run in your terminal to verify teleoperation (30s test):")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-teleoperate \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.calibration_dir=/app/calibration --robot.id=alpha_follower \\')
print(f'     --teleop.type=so101_leader --teleop.port=/dev/ttyACM0 \\')
print(f'     --teleop.calibration_dir=/app/calibration --teleop.id=alpha_leader"')
print()
print("Expected: follower arm mirrors leader arm in real time.")
print("If follower is jerky or lags: re-run calibration.")

input("\nPress Enter when teleoperation is confirmed working: ")

## 6. Record Reference Animations

**Reference animations** are 3–5 carefully executed operator demonstrations stored on HuggingFace Hub. Students replay them before coaching so they know what a good demonstration looks like.

Record each task separately. Take your time — these are the gold standard.

### 6a. Touch Task Reference

In [ ]:
TOUCH_REF_REPO   = f"{HF_USER}/soarm101-{TASK_TOUCH_SLUG}-reference"
TOUCH_REF_EPISODES = 5

print(f"Recording {TOUCH_REF_EPISODES} reference episodes for: {TASK_TOUCH_DESC}")
print(f"Dataset: {TOUCH_REF_REPO}")
print()
print("Tips for quality reference animations:")
print("  - Smooth, deliberate motions — no jerky moves")
print("  - Start and end at the same home position")
print("  - Touch the object clearly, hold for 1s, then return home")
print("  - All 5 episodes should look nearly identical")
print()

print("Run in your terminal:")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   --device=/dev/video0 --device=/dev/video2 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   -e HF_TOKEN={HF_TOKEN} \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-record \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --teleop.type=so101_leader --teleop.port=/dev/ttyACM0 \\')
print(f'     --teleop.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={TOUCH_REF_REPO} \\')
print(f'     --dataset.num_episodes={TOUCH_REF_EPISODES} \\')
print(f"     --dataset.task=\'Touch: {TASK_TOUCH_DESC}\' \\'")
print(f'     --dataset.push_to_hub=true"')

input("\nPress Enter when touch reference recording is complete: ")
print(f"Reference saved: https://huggingface.co/datasets/{TOUCH_REF_REPO}")

### 6b. Pick and Place Task Reference

In [ ]:
PICK_REF_REPO    = f"{HF_USER}/soarm101-{TASK_PICK_SLUG}-reference"
PICK_REF_EPISODES = 5

print(f"Recording {PICK_REF_EPISODES} reference episodes for: {TASK_PICK_DESC}")
print(f"Dataset: {PICK_REF_REPO}")
print()
print("Tips for quality reference animations:")
print("  - Approach the object from above, not from the side")
print("  - Close gripper fully before lifting")
print("  - Lift high enough to clear obstacles before moving laterally")
print("  - Release cleanly — open gripper fully above the target")
print("  - All 5 episodes should land the block in the same spot")
print()

print("Run in your terminal:")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   --device=/dev/video0 --device=/dev/video2 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   -e HF_TOKEN={HF_TOKEN} \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-record \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --teleop.type=so101_leader --teleop.port=/dev/ttyACM0 \\')
print(f'     --teleop.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={PICK_REF_REPO} \\')
print(f'     --dataset.num_episodes={PICK_REF_EPISODES} \\')
print(f"     --dataset.task=\'Pick and Place: {TASK_PICK_DESC}\' \\'")
print(f'     --dataset.push_to_hub=true"')

input("\nPress Enter when pick-and-place reference recording is complete: ")
print(f"Reference saved: https://huggingface.co/datasets/{PICK_REF_REPO}")

## 7. Replay Reference Animation (Verify Quality)

Replay episode 0 of each reference dataset on the follower arm. Watch the motion — if it looks wrong, re-record.

In [ ]:
def replay_reference(repo_id: str, episode: int = 0):
    print(f"Replaying episode {episode} from {repo_id}")
    print(f"Watch the follower arm — it should reproduce the reference motion.")
    print()
    print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
    print(f'  "balena run -it --privileged \\')
    print(f'   --device=/dev/ttyACM1 \\')
    print(f'   -v /mnt/data/calibration:/app/calibration \\')
    print(f'   -v /mnt/data/datasets:/app/data \\')
    print(f'   {PI_IMAGE} \\')
    print(f'   lerobot-replay \\')
    print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
    print(f'     --robot.id=alpha_follower \\')
    print(f'     --robot.calibration_dir=/app/calibration \\')
    print(f'     --dataset.repo_id={repo_id} \\')
    print(f'     --dataset.episode={episode} \\')
    print(f'     --play_sounds=false"')
    print()

replay_reference(TOUCH_REF_REPO, episode=0)
input("Press Enter after watching touch replay: ")
print()

replay_reference(PICK_REF_REPO, episode=0)
input("Press Enter after watching pick-and-place replay: ")

## Benchmark: Save Timing

In [ ]:
_bench["timings"]["total_s"] = round(time.monotonic() - _t0, 1)
_bench["completed_at"] = datetime.utcnow().isoformat()
_bench["reference_datasets"] = {
    "touch":    TOUCH_REF_REPO,
    "pick_place": PICK_REF_REPO,
}

results_dir = Path("..") / "bench" / "results"
results_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
bench_path = results_dir / f"00_arm_setup_{ts}.json"
bench_path.write_text(json.dumps(_bench, indent=2))
print(json.dumps(_bench, indent=2))
print(f"\nBenchmark saved: {bench_path}")